In [2]:
import ee
import json
import math
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import base64
from shapely.geometry import Point
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment
from PIL import Image
from io import BytesIO

# Initialize Earth Engine
try:
    ee.Initialize(project="gsapp-map")
except Exception:
    ee.Authenticate()
    ee.Initialize(project="gsapp-map")

In [3]:
# Configuration
SQUARE_VALUES = [64, 144, 256, 576, 900, 1024]  # Perfect squares for 2D (8², 12², 16², 24², 30²)
CUBE_VALUES = [25, 64, 125, 216, 512, 1000]    # Perfect cubes for 3D (4³, 5³, 6³, 8³, 10³)
N_VALUES = sorted(set(SQUARE_VALUES + CUBE_VALUES))  # Union: precompute all unique values

EMBED_PIXELS = 64
EMBED_SCALE = 10
YEAR = 2024

GEOJSON_PATH = "../dimension-reduction/data/2000_sampled_classified_embeddings.geojson"
OUTPUT_DIR = "data"
TILES_DIR = "tiles"

os.makedirs(TILES_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Will precompute for N values: {N_VALUES}")
print(f"  2D (squares): {SQUARE_VALUES}")
print(f"  3D (cubes): {CUBE_VALUES}")

Will precompute for N values: [25, 64, 125, 144, 216, 256, 512, 576, 900, 1000, 1024]
  2D (squares): [64, 144, 256, 576, 900, 1024]
  3D (cubes): [25, 64, 125, 216, 512, 1000]


In [4]:
# Load GeoJSON
print(f"Loading GeoJSON from {GEOJSON_PATH} ...")

with open(GEOJSON_PATH, "r") as f:
    geojson_data = json.load(f)

# Take maximum N needed (1000)
features = geojson_data["features"][:max(N_VALUES)]
data = []

for i, feat in enumerate(features):
    props = feat["properties"]
    coords = feat["geometry"]["coordinates"]
    
    data.append({
        "index": i,
        "lon": coords[0],
        "lat": coords[1],
        "classification": props.get("classification"),
        "subregion_name": props.get("subregion_name", "Unknown"),
        "umap_1d_x": props.get("umap_1d_x"),
        "umap_2d_x": props.get("umap_2d_x"),
        "umap_2d_y": props.get("umap_2d_y"),
        "umap_3d_x": props.get("umap_3d_x"),
        "umap_3d_y": props.get("umap_3d_y"),
        "umap_3d_z": props.get("umap_3d_z"),
        "tsne_1d_x": props.get("tsne_1d_x"),
        "tsne_2d_x": props.get("tsne_2d_x"),
        "tsne_2d_y": props.get("tsne_2d_y"),
        "tsne_3d_x": props.get("tsne_3d_x"),
        "tsne_3d_y": props.get("tsne_3d_y"),
        "tsne_3d_z": props.get("tsne_3d_z"),
    })

gdf = gpd.GeoDataFrame(
    data, geometry=[Point(d["lon"], d["lat"]) for d in data], crs="EPSG:4326"
)

print(f"Loaded {len(gdf)} samples")

Loading GeoJSON from ../dimension-reduction/data/2000_sampled_classified_embeddings.geojson ...
Loaded 1024 samples


In [5]:
gdf.head()

,index,lon,lat,classification,subregion_name,umap_1d_x,umap_2d_x,umap_2d_y,umap_3d_x,umap_3d_y,umap_3d_z,tsne_1d_x,tsne_2d_x,tsne_2d_y,tsne_3d_x,tsne_3d_y,tsne_3d_z,geometry
0,0,-131.184025,59.611055,126,Northern America,16.784231,-40.799011,-2.798702,-45.941341,4.641312,-26.571562,-49.619076,-26.663736,13.954807,-16.747684,38.547501,29.680862,POINT (-131.18402 59.61106)
1,1,-62.623884,9.017345,30,Latin America and the Caribbean,0.639102,10.030628,43.123466,6.111301,49.059601,24.983414,17.551109,19.866989,-37.949833,36.957924,-40.670971,20.553169,POINT (-62.62388 9.01735)
2,2,11.678798,-3.611274,30,Sub-Saharan Africa,-12.320284,48.158806,-12.502239,36.104492,-3.900765,27.594849,49.514023,40.090767,-2.841045,-2.776424,-47.970524,-25.487997,POINT (11.6788 -3.61127)
3,3,10.361996,45.732221,115,Southern Europe,13.234972,-24.740774,-2.040253,-35.001968,-5.382343,-0.889820,-24.178932,-10.349129,7.788277,-24.244860,5.883209,0.029480,POINT (10.362 45.73222)
4,4,32.748271,58.383271,126,Eastern Europe,21.671848,-27.601662,-13.733841,-35.775017,-11.860596,-15.688488,-22.066343,-14.942474,15.959877,-30.494827,11.660652,16.022049,POINT (32.74827 58.38327)


In [6]:
# Helper function to arrange points in grid using Hungarian algorithm
def arrange_to_grid_1d(values, n_points):
    """Arrange 1D values to 1D grid positions (0 to n_points-1)"""
    grid_pos = np.arange(n_points).reshape(-1, 1)
    values_2d = values.reshape(-1, 1)
    
    cost = cdist(values_2d, grid_pos)
    r_idx, c_idx = linear_sum_assignment(cost)
    
    result = np.zeros(n_points)
    result[r_idx] = grid_pos[c_idx].flatten()
    return result


def arrange_to_grid_2d(coords_2d, n_points):
    """Arrange 2D coords to square grid"""
    grid_side = int(math.ceil(math.sqrt(n_points)))
    grid_x = np.linspace(0, grid_side - 1, grid_side)
    grid_y = np.linspace(0, grid_side - 1, grid_side)
    grid_coords = np.array(np.meshgrid(grid_x, grid_y)).T.reshape(-1, 2)[:n_points]
    
    cost = cdist(coords_2d, grid_coords)
    r_idx, c_idx = linear_sum_assignment(cost)
    
    result = np.zeros((n_points, 2))
    result[r_idx] = grid_coords[c_idx]
    return result


def arrange_to_grid_3d(coords_3d, n_points):
    """Arrange 3D coords to cube grid"""
    cube_side = int(math.ceil(n_points ** (1/3)))
    grid_x = np.linspace(0, cube_side - 1, cube_side)
    grid_y = np.linspace(0, cube_side - 1, cube_side)
    grid_z = np.linspace(0, cube_side - 1, cube_side)
    grid_coords = np.array(np.meshgrid(grid_x, grid_y, grid_z)).T.reshape(-1, 3)[:n_points]
    
    cost = cdist(coords_3d, grid_coords)
    r_idx, c_idx = linear_sum_assignment(cost)
    
    result = np.zeros((n_points, 3))
    result[r_idx] = grid_coords[c_idx]
    return result

print("Grid arrangement functions defined")

Grid arrangement functions defined


In [7]:
# Generate grid positions for each N value
all_gdfs = {}

for n in N_VALUES:
    print(f"\n=== Processing N = {n} ===")
    
    # Get subset of data
    gdf_subset = gdf.iloc[:n].copy()
    
    # UMAP 1D
    print(f"Arranging UMAP 1D for {n} samples...")
    umap_1d = gdf_subset["umap_1d_x"].values.astype(np.float32)
    grid_1d = arrange_to_grid_1d(umap_1d, n)
    gdf_subset[f"grid_umap_1d_x"] = grid_1d
    
    # UMAP 2D
    print(f"Arranging UMAP 2D for {n} samples...")
    umap_2d = gdf_subset[["umap_2d_x", "umap_2d_y"]].values.astype(np.float32)
    grid_2d = arrange_to_grid_2d(umap_2d, n)
    gdf_subset[f"grid_umap_2d_x"] = grid_2d[:, 0]
    gdf_subset[f"grid_umap_2d_y"] = grid_2d[:, 1]
    
    # UMAP 3D
    print(f"Arranging UMAP 3D for {n} samples...")
    umap_3d = gdf_subset[["umap_3d_x", "umap_3d_y", "umap_3d_z"]].values.astype(np.float32)
    grid_3d = arrange_to_grid_3d(umap_3d, n)
    gdf_subset[f"grid_umap_3d_x"] = grid_3d[:, 0]
    gdf_subset[f"grid_umap_3d_y"] = grid_3d[:, 1]
    gdf_subset[f"grid_umap_3d_z"] = grid_3d[:, 2]
    
    # t-SNE 1D
    print(f"Arranging t-SNE 1D for {n} samples...")
    tsne_1d = gdf_subset["tsne_1d_x"].values.astype(np.float32)
    grid_1d = arrange_to_grid_1d(tsne_1d, n)
    gdf_subset[f"grid_tsne_1d_x"] = grid_1d
    
    # t-SNE 2D
    print(f"Arranging t-SNE 2D for {n} samples...")
    tsne_2d = gdf_subset[["tsne_2d_x", "tsne_2d_y"]].values.astype(np.float32)
    grid_2d = arrange_to_grid_2d(tsne_2d, n)
    gdf_subset[f"grid_tsne_2d_x"] = grid_2d[:, 0]
    gdf_subset[f"grid_tsne_2d_y"] = grid_2d[:, 1]
    
    # t-SNE 3D
    print(f"Arranging t-SNE 3D for {n} samples...")
    tsne_3d = gdf_subset[["tsne_3d_x", "tsne_3d_y", "tsne_3d_z"]].values.astype(np.float32)
    grid_3d = arrange_to_grid_3d(tsne_3d, n)
    gdf_subset[f"grid_tsne_3d_x"] = grid_3d[:, 0]
    gdf_subset[f"grid_tsne_3d_y"] = grid_3d[:, 1]
    gdf_subset[f"grid_tsne_3d_z"] = grid_3d[:, 2]
    
    all_gdfs[n] = gdf_subset

print("\nGrid positions generated for all N values")


=== Processing N = 25 ===
Arranging UMAP 1D for 25 samples...
Arranging UMAP 2D for 25 samples...
Arranging UMAP 3D for 25 samples...
Arranging t-SNE 1D for 25 samples...
Arranging t-SNE 2D for 25 samples...
Arranging t-SNE 3D for 25 samples...

=== Processing N = 64 ===
Arranging UMAP 1D for 64 samples...
Arranging UMAP 2D for 64 samples...
Arranging UMAP 3D for 64 samples...
Arranging t-SNE 1D for 64 samples...
Arranging t-SNE 2D for 64 samples...
Arranging t-SNE 3D for 64 samples...

=== Processing N = 125 ===
Arranging UMAP 1D for 125 samples...
Arranging UMAP 2D for 125 samples...
Arranging UMAP 3D for 125 samples...
Arranging t-SNE 1D for 125 samples...
Arranging t-SNE 2D for 125 samples...
Arranging t-SNE 3D for 125 samples...

=== Processing N = 144 ===
Arranging UMAP 1D for 144 samples...
Arranging UMAP 2D for 144 samples...
Arranging UMAP 3D for 144 samples...
Arranging t-SNE 1D for 144 samples...
Arranging t-SNE 2D for 144 samples...
Arranging t-SNE 3D for 144 samples...

=

In [ ]:
# Download satellite patches (only need max N)
max_n = max(N_VALUES)
print(f"\nDownloading satellite patches for {max_n} locations...")

# Import for concurrent downloads
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

# Two datasets: embedding and true color satellite
embedding_dataset = ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")
satellite_dataset = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
landsat_dataset = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")

# Pre-filter datasets once
embedding_filtered = embedding_dataset.filterDate(f"{YEAR}-01-01", f"{YEAR + 1}-01-01")

# Generate all band combinations (A0-A1-A2, A1-A2-A3, ..., A61-A62-A63)
# Total of 62 combinations (bands are 0-indexed: A00 to A63)
band_combinations = [(i, i+1, i+2) for i in range(0, 62)]
print(f"Will generate {len(band_combinations)} band combinations per location")

def download_image(url, filepath):
    """Download an image from URL to filepath"""
    try:
        response = requests.get(url, timeout=60)
        if response.status_code == 200:
            with open(filepath, 'wb') as f:
                f.write(response.content)
            return True
        return False
    except Exception as e:
        return False

def get_rgb_image_url(region, year):
    """Try multiple strategies to get RGB satellite image URL"""
    # Strategy 1: Try Sentinel-2 with cloud filter
    try:
        rgb_img = (
            satellite_dataset
            .filterDate(f"{year}-01-01", f"{year + 1}-01-01")
            .filterBounds(region)
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
            .median()
        )
        band_names = rgb_img.bandNames().getInfo()
        if len(band_names) > 0:
            rgb_img = rgb_img.select(['B4', 'B3', 'B2']).visualize(min=0, max=3000)
            return rgb_img.getThumbURL({
                'region': region,
                'dimensions': f'{EMBED_PIXELS}x{EMBED_PIXELS}',
                'format': 'png'
            })
    except Exception:
        pass
    
    # Strategy 2: Try Sentinel-2 without cloud filter
    try:
        rgb_img = (
            satellite_dataset
            .filterDate(f"{year}-01-01", f"{year + 1}-01-01")
            .filterBounds(region)
            .median()
        )
        band_names = rgb_img.bandNames().getInfo()
        if len(band_names) > 0:
            rgb_img = rgb_img.select(['B4', 'B3', 'B2']).visualize(min=0, max=3000)
            return rgb_img.getThumbURL({
                'region': region,
                'dimensions': f'{EMBED_PIXELS}x{EMBED_PIXELS}',
                'format': 'png'
            })
    except Exception:
        pass
    
    return None

def process_location(idx, row):
    """Process a single location - generate URLs and download"""
    tile_dir = os.path.join(TILES_DIR, f"{idx:04d}")
    os.makedirs(tile_dir, exist_ok=True)
    
    point = ee.Geometry.Point(row.lon, row.lat)
    region = point.buffer(EMBED_SCALE * EMBED_PIXELS / 2).bounds()
    
    # Check what files we need
    files_needed = []
    
    # Check all band combinations
    for b1, b2, b3 in band_combinations:
        filepath = os.path.join(tile_dir, f"{idx:04d}_A{b1}_A{b2}_A{b3}.png")
        if not os.path.exists(filepath):
            files_needed.append(('embed', b1, b2, b3, filepath))
    
    # Check satellite RGB
    rgb_filepath = os.path.join(tile_dir, f"{idx:04d}_satellite.png")
    if not os.path.exists(rgb_filepath):
        files_needed.append(('rgb', None, None, None, rgb_filepath))
    
    if len(files_needed) == 0:
        return idx, 0, 0  # idx, files_downloaded, files_skipped
    
    # Generate URLs for all needed files (with retry logic)
    urls_to_download = []
    failed_files = []
    
    def generate_embed_url_with_retry(b1, b2, b3, filepath, retries=3):
        """Generate URL with retry logic"""
        for attempt in range(retries):
            try:
                embed_img = (
                    embedding_filtered
                    .filterBounds(region)
                    .mosaic()
                    .select([f"A{b1:02d}", f"A{b2:02d}", f"A{b3:02d}"])
                )
                
                url = embed_img.getThumbURL({
                    'region': region,
                    'dimensions': f'{EMBED_PIXELS}x{EMBED_PIXELS}',
                    'format': 'png',
                    'min': -0.3,
                    'max': 0.3
                })
                return (url, filepath)
            except Exception as e:
                if attempt == retries - 1:
                    return None
                time.sleep(0.5 * (attempt + 1))  # Exponential backoff
        return None
    
    # Generate embedding URLs sequentially (more reliable than parallel)
    # Parallel URL generation was causing too many Earth Engine API failures
    for file_type, b1, b2, b3, filepath in files_needed:
        if file_type == 'embed':
            result = generate_embed_url_with_retry(b1, b2, b3, filepath)
            if result:
                urls_to_download.append(result)
            else:
                failed_files.append(f"A{b1}_A{b2}_A{b3}")
    
    # Generate RGB URL (if needed)
    for file_type, b1, b2, b3, filepath in files_needed:
        if file_type == 'rgb':
            url = get_rgb_image_url(region, YEAR)
            if url:
                urls_to_download.append((url, filepath))
            else:
                failed_files.append("satellite")
    
    # Download all files in parallel
    download_success = 0
    if len(urls_to_download) > 0:
        with ThreadPoolExecutor(max_workers=20) as executor:
            download_futures = {executor.submit(download_image, url, filepath): filepath 
                              for url, filepath in urls_to_download}
            for future in as_completed(download_futures):
                if future.result():
                    download_success += 1
    
    # Log if there were failures
    if len(failed_files) > 0:
        print(f"    Location {idx}: Failed to generate URLs for {len(failed_files)} files: {', '.join(failed_files[:5])}")
    
    return idx, download_success, len(files_needed) - download_success

# Process locations with retry logic - multiple passes until all files are downloaded
print("Processing locations with retry logic...")
start_time = time.time()

MAX_PASSES = 5  # Maximum number of retry passes
locations_to_process = list(gdf.iterrows())

for pass_num in range(1, MAX_PASSES + 1):
    print(f"\n=== Pass {pass_num}/{MAX_PASSES} - Processing {len(locations_to_process)} locations ===")
    
    if len(locations_to_process) == 0:
        print("All locations complete!")
        break
    
    with ThreadPoolExecutor(max_workers=3) as executor:
        futures = {}
        for idx, row in locations_to_process:
            future = executor.submit(process_location, idx, row)
            futures[future] = (idx, row)
        
        completed = 0
        pass_downloaded = 0
        pass_failed = 0
        locations_with_failures = []
        
        for future in as_completed(futures):
            idx, row = futures[future]
            result_idx, downloaded, failed = future.result()
            completed += 1
            pass_downloaded += downloaded
            pass_failed += failed
            
            # Track locations that still have failures for next pass
            if failed > 0:
                locations_with_failures.append((idx, row))
            
            if completed % 10 == 0 or failed > 0:
                elapsed = time.time() - start_time
                rate = elapsed / completed if completed > 0 else 0
                remaining = (len(locations_to_process) - completed) * rate
                print(f"  {completed}/{len(locations_to_process)} - Location {idx}: {downloaded} downloaded, {failed} failed | "
                      f"Est remaining: {remaining/60:.1f}min")
    
    print(f"Pass {pass_num} complete: {pass_downloaded} files downloaded, {pass_failed} files failed")
    
    # Prepare next pass with only failed locations
    locations_to_process = locations_with_failures
    
    if len(locations_to_process) > 0:
        print(f"  {len(locations_to_process)} locations still have failures, will retry...")
        time.sleep(2)  # Brief pause between passes

total_time = time.time() - start_time
print(f"\n{'='*60}")
print(f"All passes complete in {total_time/60:.1f} minutes")

# Final verification - count missing files
print("\nVerifying downloads...")
total_missing = 0
for idx, row in gdf.iterrows():
    tile_dir = os.path.join(TILES_DIR, f"{idx:04d}")
    missing = 0
    for b1, b2, b3 in band_combinations:
        filepath = os.path.join(tile_dir, f"{idx:04d}_A{b1}_A{b2}_A{b3}.png")
        if not os.path.exists(filepath):
            missing += 1
    rgb_filepath = os.path.join(tile_dir, f"{idx:04d}_satellite.png")
    if not os.path.exists(rgb_filepath):
        missing += 1
    if missing > 0:
        total_missing += missing
        if total_missing <= 100:  # Only print first 100 to avoid spam
            print(f"  Location {idx}: {missing} files still missing")

if total_missing == 0:
    print("✓ All files successfully downloaded!")
else:
    print(f"\n⚠ {total_missing} total files still missing after all retries")


Will generate 62 band combinations per location
Processing locations with retry logic...

=== Pass 1/5 - Processing 1024 locations ===
  10/1024 - Location 10: 0 downloaded, 0 failed | Est remaining: 0.1min
  20/1024 - Location 21: 0 downloaded, 0 failed | Est remaining: 3.1min
  30/1024 - Location 30: 5 downloaded, 0 failed | Est remaining: 5.2min
  40/1024 - Location 36: 6 downloaded, 0 failed | Est remaining: 6.4min
  50/1024 - Location 47: 6 downloaded, 13 failed | Est remaining: 7.2min
  52/1024 - Location 50: 28 downloaded, 1 failed | Est remaining: 7.6min
  54/1024 - Location 51: 20 downloaded, 2 failed | Est remaining: 8.4min
  55/1024 - Location 54: 18 downloaded, 2 failed | Est remaining: 8.4min
  59/1024 - Location 56: 23 downloaded, 2 failed | Est remaining: 9.3min
  60/1024 - Location 60: 22 downloaded, 1 failed | Est remaining: 9.5min
  64/1024 - Location 62: 25 downloaded, 1 failed | Est remaining: 10.5min
  65/1024 - Location 65: 9 downloaded, 2 failed | Est remaining:

In [ ]:
# Save GeoJSON files for each N value
print("\nSaving GeoJSON files...")

for n, gdf_subset in all_gdfs.items():
    # Add tile directory paths to GeoDataFrame
    gdf_subset["tile_dir"] = [f"tiles/{i:04d}" for i in gdf_subset["index"]]
    
    # Save to GeoJSON
    output_path = os.path.join(OUTPUT_DIR, f"web_grid_data_{n}.geojson")
    gdf_subset.to_file(output_path, driver="GeoJSON")
    print(f"  Saved {len(gdf_subset)} features to {output_path}")

print("\nAll files saved successfully!")
print(f"N values: {N_VALUES}")


Saving GeoJSON files...
  Saved 25 features to data/web_grid_data_25.geojson
  Saved 64 features to data/web_grid_data_64.geojson
  Saved 125 features to data/web_grid_data_125.geojson
  Saved 144 features to data/web_grid_data_144.geojson
  Saved 216 features to data/web_grid_data_216.geojson
  Saved 256 features to data/web_grid_data_256.geojson
  Saved 512 features to data/web_grid_data_512.geojson
  Saved 576 features to data/web_grid_data_576.geojson
  Saved 900 features to data/web_grid_data_900.geojson
  Saved 1000 features to data/web_grid_data_1000.geojson
  Saved 1024 features to data/web_grid_data_1024.geojson

All files saved successfully!
N values: [25, 64, 125, 144, 216, 256, 512, 576, 900, 1000, 1024]
